In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

# 1. Definition of the original aperiodic signal x[n] of length L = 15
L = 15
n_signal = np.arange(L)
x = np.exp(-0.1 * n_signal) * np.cos(2 * np.pi * 0.1 * n_signal)

# 2. Calculation of the "True" Continuous DTFT of the original signal x[n]
omega_dense = np.linspace(0, 2 * np.pi, 2000)
X_dtft_original = np.array([np.sum(x * np.exp(-1j * w * n_signal)) for w in omega_dense])

# 3. Update and plotting function with Reconstructed Spectrum and external legend
def plot_dft_sampling_with_distortion(N):
    actual_N = N  
        
    # Calculation of periodic extension xi[n]
    xi_n = np.zeros(actual_N)
    for n in range(actual_N):
        m_indices = np.arange(-5, 6)
        for m in m_indices:
            idx = n - m * actual_N
            if 0 <= idx < L:
                xi_n[n] += x[idx]

    # Calculation of DFT of size actual_N
    X_k = np.fft.fft(xi_n, actual_N)
    frequencies = np.linspace(0, 2 * np.pi, actual_N, endpoint=False)

    # Calculation of Reconstructed Spectrum (DTFT from DFT samples via interpolation formula)
    X_reconstructed = np.zeros_like(omega_dense, dtype=complex)
    for k, X_val in enumerate(X_k):
        omega_k = 2 * np.pi * k / actual_N
        arg = omega_dense - omega_k
        with np.errstate(divide='ignore', invalid='ignore'):
            P = np.sin(arg * actual_N / 2) / (actual_N * np.sin(arg / 2))
            P[np.isnan(P)] = 1.0 
        X_reconstructed += X_val * P * np.exp(-1j * arg * (actual_N - 1) / 2)

    # Plotting with optimized height (figsize=(12, 8)) to avoid scrolling
    fig, axes = plt.subplots(3, 1, figsize=(12, 8))

    # --- Plot 1: Original Signal x[n] ---
    axes[0].stem(n_signal, x, basefmt=" ", linefmt='b-', markerfmt='bo')
    axes[0].set_title(f'Original Non-periodic Signal $x[n]$ (Length $L = {L}$)', fontsize=10, fontweight='bold')
    axes[0].set_xlabel('$n$', fontsize=9)
    axes[0].set_ylabel('$x[n]$', fontsize=9)
    axes[0].grid(True, linestyle='--', alpha=0.6)

    # --- Plot 2: Periodic Extension xi[n] ---
    if actual_N >= L:
        status_text = r"No Aliasing ($N \geq L$) — Clean separation"
        color_line = 'g-'
        marker_color = 'go'
        title_color = 'darkgreen'
    else:
        status_text = r"Aliasing Present ($N < L$) — Overlap / Folding!"
        color_line = 'm-'
        marker_color = 'mo'
        title_color = 'darkmagenta'
    
    n_extended = np.arange(-actual_N, 2 * actual_N)
    xi_extended = np.tile(xi_n, 3)
    
    axes[1].stem(n_extended, xi_extended, basefmt=" ", linefmt=color_line, markerfmt=marker_color)
    axes[1].set_title(rf'Periodic Extension $\xi[n]$ | Period $N = {actual_N}$ — [{status_text}]', 
                      fontsize=10, fontweight='bold', color=title_color)
    axes[1].set_xlabel('$n$', fontsize=9)
    axes[1].set_ylabel(r'$\xi[n]$', fontsize=9)
    axes[1].grid(True, linestyle='--', alpha=0.6)

    # --- Plot 3: Spectrum Comparison & Distortion (Legend placed outside on the right) ---
    axes[2].plot(omega_dense, np.abs(X_dtft_original), 'k-', linewidth=2, label=r'Original DTFT $|\mathcal{X}(e^{j\omega})|$')
    axes[2].plot(omega_dense, np.abs(X_reconstructed), 'r--', linewidth=2, label=r'Reconstructed DTFT from DFT')
    axes[2].stem(frequencies, np.abs(X_k), basefmt=" ", linefmt='b-', markerfmt='bo', label=rf'DFT Samples $\mathcal{{X}}[k]$ ($N={actual_N}$)')
    axes[2].set_title('Spectrum Distortion Due to Aliasing (Comparison of Original vs Reconstructed)', fontsize=10, fontweight='bold')
    axes[2].set_xlabel(r'Frequency $\omega$ (rad)', fontsize=9)
    axes[2].set_ylabel('Magnitude', fontsize=9)
    
    # Move legend outside to the right
    axes[2].legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9)
    axes[2].grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

# 4. Explanatory text widget placed right above the slider (using raw string r""" ... """)
instructions_html = HTML(r"""
<div style="background-color: #f8f9fa; padding: 10px; border-left: 4px solid #007bff; margin-bottom: 10px; font-family: Arial, sans-serif; font-size: 13px;">
    <b>Interactive Exploration: Frequency Domain Sampling & Aliasing</b><br>
    Use the slider below to adjust the DFT size <b>$N$</b>. 
    <ul>
        <li><b>When $N \ge L$ ($N \ge 15$):</b> The periodic extension $\xi[n]$ has clear separation with no overlap, and the reconstructed spectrum perfectly matches the true continuous DTFT.</li>
        <li><b>When $N < L$ ($N < 15$):</b> Time-domain aliasing occurs due to overlapping periods, leading to visible spectrum distortion in the reconstructed red dashed curve.</li>
    </ul>
</div>
""")
display(instructions_html)

# 5. Creation of the Slider Control
n_slider = widgets.IntSlider(
    value=32,            
    min=6,               # Smaller than L=15, immediately triggers aliasing
    max=64,              
    step=1,
    description='N (DFT Size):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

widgets.interactive(plot_dft_sampling_with_distortion, N=n_slider)